# PAP — Capítulo 4 — Lógica de Agentes
## Hands-On — Etapa 2 — Gabarito: PetTech

**Objetivo:** adaptar a arquitetura do agente de manutenção para um novo domínio, preservando a separação entre decisão da LLM e controle determinístico do Python.

Neste gabarito, o agente realiza o **pré-agendamento de serviços estéticos** para cães e gatos. Ele pode:

- calcular o orçamento;
- verificar a disponibilidade da agenda;
- registrar o pré-agendamento depois da confirmação do cliente.

As regras de preço, agenda, validação e registro permanecem no Python. A LLM participa da conversa e escolhe quando solicitar cada capacidade.

## 1. Preparação do ambiente

No Google Colab, adicione sua chave da Gemini API em **Secrets** com o nome `GOOGLE_API_KEY`.

In [ ]:
%pip install -q -U "google-genai>=2.3.0,<3" "pydantic>=2,<3"

In [ ]:
from google import genai
from google.colab import userdata
from pydantic import BaseModel, ConfigDict, Field, ValidationError
from typing import Literal

import json
import logging
import unicodedata

api_key = userdata.get("GOOGLE_API_KEY")

if not api_key:
    raise ValueError(
        "Adicione GOOGLE_API_KEY aos Secrets do Google Colab antes de executar."
    )

MODELO = "gemini-3.6-flash"

client = genai.Client(api_key=api_key)

logging.basicConfig(
    level=logging.INFO,
    format="%(levelname)s: %(message)s"
)

print("Ambiente preparado.")

## 2. Dados do PetTech

Preço e disponibilidade são regras do domínio. A LLM não deve inventar esses valores.

In [ ]:
precos = {
    "banho": {
        "pequeno": 45.0,
        "medio": 60.0,
        "grande": 80.0
    },
    "tosa": {
        "pequeno": 35.0,
        "medio": 50.0,
        "grande": 65.0
    },
    "hidratacao": {
        "pequeno": 20.0,
        "medio": 30.0,
        "grande": 40.0
    }
}

agenda = {
    "segunda-feira": {
        "manha": True,
        "tarde": False
    },
    "quarta-feira": {
        "manha": False,
        "tarde": True
    },
    "sabado": {
        "manha": True,
        "tarde": False
    }
}

pre_agendamentos = []

print("Serviços:", list(precos.keys()))
print("Dias disponíveis:", list(agenda.keys()))

## 3. Capacidades da aplicação

As três funções abaixo representam as capacidades reais do sistema.

`calcular_orcamento()` e `verificar_disponibilidade()` apenas consultam regras.

`registrar_pre_agendamento()` altera o estado da aplicação ao inserir uma ficha na lista `pre_agendamentos`.

In [ ]:
def calcular_orcamento(porte, servicos):
    total = 0.0

    for servico in servicos:
        total += precos[servico][porte]

    return round(total, 2)


def verificar_disponibilidade(dia, periodo):
    return {
        "dia": dia,
        "periodo": periodo,
        "disponivel": agenda[dia][periodo]
    }


def registrar_pre_agendamento(
    nome_pet,
    especie,
    porte,
    servicos,
    dia,
    periodo,
    valor_total
):
    ficha = {
        "nome_pet": nome_pet,
        "especie": especie,
        "porte": porte,
        "servicos": servicos,
        "dia": dia,
        "periodo": periodo,
        "valor_total": valor_total
    }

    pre_agendamentos.append(ficha)

    return ficha

## 4. Contratos com Pydantic

Os contratos definem os formatos aceitos antes da execução das funções.

In [ ]:
Servico = Literal["banho", "tosa", "hidratacao"]
Porte = Literal["pequeno", "medio", "grande"]
Especie = Literal["cao", "gato"]
Periodo = Literal["manha", "tarde"]
Dia = Literal["segunda-feira", "quarta-feira", "sabado"]


class ArgumentosOrcamento(BaseModel):
    model_config = ConfigDict(
        strict=True,
        extra="forbid"
    )

    porte: Porte
    servicos: list[Servico]


class ArgumentosDisponibilidade(BaseModel):
    model_config = ConfigDict(
        strict=True,
        extra="forbid"
    )

    dia: Dia
    periodo: Periodo


class FichaAtendimento(BaseModel):
    model_config = ConfigDict(
        strict=True,
        extra="forbid"
    )

    nome_pet: str = Field(min_length=1)
    especie: Especie
    porte: Porte
    servicos: list[Servico]
    dia: Dia
    periodo: Periodo
    valor_total: float = Field(ge=0)


class ArgumentosPreAgendamento(FichaAtendimento):
    pass

## 5. Declaração das tools

As descrições informam à LLM o que cada capacidade faz. Elas não executam as funções.

In [ ]:
calcular_orcamento_tool = {
    "type": "function",
    "name": "calcular_orcamento",
    "description": (
        "Calcula o valor total dos serviços estéticos de acordo "
        "com o porte do pet. Use sempre esta tool para obter preços."
    ),
    "parameters": {
        "type": "object",
        "properties": {
            "porte": {
                "type": "string",
                "enum": ["pequeno", "medio", "grande"]
            },
            "servicos": {
                "type": "array",
                "items": {
                    "type": "string",
                    "enum": ["banho", "tosa", "hidratacao"]
                }
            }
        },
        "required": ["porte", "servicos"]
    }
}


verificar_disponibilidade_tool = {
    "type": "function",
    "name": "verificar_disponibilidade",
    "description": (
        "Verifica se o dia e o período informados estão disponíveis "
        "na agenda do PetTech. Nunca invente disponibilidade."
    ),
    "parameters": {
        "type": "object",
        "properties": {
            "dia": {
                "type": "string",
                "enum": [
                    "segunda-feira",
                    "quarta-feira",
                    "sabado"
                ]
            },
            "periodo": {
                "type": "string",
                "enum": ["manha", "tarde"]
            }
        },
        "required": ["dia", "periodo"]
    }
}


registrar_pre_agendamento_tool = {
    "type": "function",
    "name": "registrar_pre_agendamento",
    "description": (
        "Registra a ficha final do pré-agendamento. "
        "Só deve ser solicitada depois que orçamento e disponibilidade "
        "forem verificados e o cliente confirmar explicitamente os dados."
    ),
    "parameters": {
        "type": "object",
        "properties": {
            "nome_pet": {"type": "string"},
            "especie": {
                "type": "string",
                "enum": ["cao", "gato"]
            },
            "porte": {
                "type": "string",
                "enum": ["pequeno", "medio", "grande"]
            },
            "servicos": {
                "type": "array",
                "items": {
                    "type": "string",
                    "enum": ["banho", "tosa", "hidratacao"]
                }
            },
            "dia": {
                "type": "string",
                "enum": [
                    "segunda-feira",
                    "quarta-feira",
                    "sabado"
                ]
            },
            "periodo": {
                "type": "string",
                "enum": ["manha", "tarde"]
            },
            "valor_total": {
                "type": "number"
            }
        },
        "required": [
            "nome_pet",
            "especie",
            "porte",
            "servicos",
            "dia",
            "periodo",
            "valor_total"
        ]
    }
}


tools = [
    calcular_orcamento_tool,
    verificar_disponibilidade_tool,
    registrar_pre_agendamento_tool
]

print("Tools:", [tool["name"] for tool in tools])

## 6. Allowlist e regras determinísticas

A aplicação não executa qualquer nome produzido pela LLM.

Além da allowlist e dos contratos, o registro final possui verificações adicionais:

- deve existir confirmação explícita do cliente;
- o preço deve coincidir com o cálculo do Python;
- o horário deve estar disponível;
- a mesma sessão não pode registrar duas vezes o mesmo atendimento.

In [ ]:
tools_permitidas = {
    "calcular_orcamento": (
        calcular_orcamento,
        ArgumentosOrcamento
    ),
    "verificar_disponibilidade": (
        verificar_disponibilidade,
        ArgumentosDisponibilidade
    ),
    "registrar_pre_agendamento": (
        registrar_pre_agendamento,
        ArgumentosPreAgendamento
    )
}


def normalizar_texto(texto):
    texto = texto.lower().strip()

    texto = "".join(
        caractere
        for caractere in unicodedata.normalize("NFD", texto)
        if unicodedata.category(caractere) != "Mn"
    )

    return texto


def confirmacao_explicita(mensagem):
    texto = normalizar_texto(mensagem)

    expressoes = [
        "confirmo",
        "sim, confirmo",
        "pode confirmar",
        "esta correto",
        "estao corretos",
        "pode registrar"
    ]

    return any(
        expressao in texto
        for expressao in expressoes
    )


def detectar_alerta_saude(mensagem):
    texto = normalizar_texto(mensagem)

    termos = [
        "machucado",
        "machucada",
        "doente",
        "apatico",
        "apatica",
        "vomitando",
        "vomito",
        "sangrando",
        "ferida",
        "febre",
        "convulsao",
        "diarreia"
    ]

    return any(
        termo in texto
        for termo in termos
    )

### Regra de segurança veterinária

O system prompt também orientará a LLM, mas o gabarito mantém uma barreira simples em Python para os casos explícitos usados na atividade.

Esse filtro não é um sistema de diagnóstico. Ele apenas interrompe o fluxo de pré-agendamento quando aparecem sinais definidos para o exercício.

In [ ]:
MENSAGEM_SAUDE = (
    "Não vou continuar o pré-agendamento estético porque a mensagem "
    "indica um possível problema de saúde. Procure atendimento com "
    "um médico veterinário para avaliar o pet."
)

### Executor das tools

O registro final é revalidado pelo Python. Mesmo que a LLM envie um preço incorreto ou tente registrar um horário indisponível, a operação deve ser recusada.

In [ ]:
def validar_registro(dados, mensagem_usuario, sessao):
    if sessao["registro_concluido"]:
        return {
            "erro": "pre_agendamento_ja_registrado"
        }

    if not confirmacao_explicita(mensagem_usuario):
        return {
            "erro": "confirmacao_ausente",
            "detalhe": (
                "O cliente ainda não confirmou explicitamente "
                "os dados do pré-agendamento."
            )
        }

    valor_correto = calcular_orcamento(
        dados["porte"],
        dados["servicos"]
    )

    if abs(dados["valor_total"] - valor_correto) > 0.001:
        return {
            "erro": "valor_incorreto",
            "valor_correto": valor_correto
        }

    disponibilidade = verificar_disponibilidade(
        dados["dia"],
        dados["periodo"]
    )

    if not disponibilidade["disponivel"]:
        return {
            "erro": "horario_indisponivel",
            "dia": dados["dia"],
            "periodo": dados["periodo"]
        }

    return None


def executar_tool(
    nome,
    argumentos,
    mensagem_usuario,
    sessao
):
    configuracao = tools_permitidas.get(nome)

    if configuracao is None:
        logging.warning("Tool recusada: %s", nome)
        return {
            "erro": "tool_nao_permitida"
        }

    funcao, modelo_argumentos = configuracao

    try:
        dados = modelo_argumentos.model_validate(
            argumentos
        )

    except ValidationError as erro:
        logging.warning(
            "Argumentos inválidos para %s",
            nome
        )
        return {
            "erro": "argumentos_invalidos",
            "detalhe": erro.errors()
        }

    argumentos_validados = dados.model_dump()

    if nome == "registrar_pre_agendamento":
        recusa = validar_registro(
            argumentos_validados,
            mensagem_usuario,
            sessao
        )

        if recusa is not None:
            logging.warning(
                "Registro recusado: %s",
                recusa
            )
            return recusa

    try:
        logging.info("Executando tool: %s", nome)

        resultado = funcao(
            **argumentos_validados
        )

        if nome == "registrar_pre_agendamento":
            sessao["registro_concluido"] = True

        return resultado

    except Exception as erro:
        logging.error(
            "Falha durante a execução de %s: %s",
            nome,
            erro
        )

        return {
            "erro": "falha_execucao"
        }

## 7. Instrução do agente

O prompt orienta o comportamento da LLM, mas não substitui as validações do Python.

In [ ]:
INSTRUCAO_SISTEMA = """
Você é o assistente virtual do Pet Shop PetTech.

Seu escopo é exclusivamente o pré-agendamento de serviços estéticos:
banho, tosa e hidratação para cães e gatos.

REGRAS:
- Nunca dê diagnóstico, tratamento ou orientação médica veterinária.
- Se houver relato de animal machucado, doente, apático ou com sintomas,
  interrompa o pré-agendamento e recomende atendimento veterinário.
- Nunca invente preços. Use calcular_orcamento.
- Nunca invente horários disponíveis. Use verificar_disponibilidade.
- Colete nome do pet, espécie, porte, serviços, dia e período.
- Se faltar informação, pergunte antes de avançar.
- Depois de obter orçamento e disponibilidade, apresente um resumo completo.
- Peça confirmação explícita do cliente antes do registro.
- Só depois de uma nova mensagem confirmando os dados solicite
  registrar_pre_agendamento.
- Se uma tool retornar erro ou recusa, considere esse resultado.
- Nunca afirme que o pré-agendamento foi registrado sem confirmação da tool.
- Solicite no máximo uma tool por interação.
- Quando houver informação suficiente e nenhuma ação pendente, responda ao cliente.
"""

## 8. Sessão e ciclo do agente

`previous_interaction_id` mantém a continuidade entre as mensagens. Assim, o cliente não precisa repetir nome do pet, porte, serviços e horário em cada turno.

In [ ]:
MAX_ITERACOES_POR_TURNO = 5


def nova_sessao():
    return {
        "previous_interaction_id": None,
        "turnos": 0,
        "acoes_executadas": [],
        "registro_concluido": False
    }


def encontrar_function_call(interaction):
    for passo in interaction.steps or []:
        if passo.type == "function_call":
            return passo

    return None

In [ ]:
def atender(mensagem_usuario, sessao):
    sessao["turnos"] += 1

    logging.info(
        "Turno %s",
        sessao["turnos"]
    )

    # Barreira determinística antes da LLM.
    if detectar_alerta_saude(mensagem_usuario):
        logging.warning(
            "Fluxo interrompido por alerta de saúde."
        )
        return MENSAGEM_SAUDE

    argumentos_interaction = {
        "model": MODELO,
        "input": mensagem_usuario,
        "tools": tools
    }

    if sessao["previous_interaction_id"] is None:
        argumentos_interaction["system_instruction"] = (
            INSTRUCAO_SISTEMA
        )
    else:
        argumentos_interaction["previous_interaction_id"] = (
            sessao["previous_interaction_id"]
        )

    interaction = client.interactions.create(
        **argumentos_interaction
    )

    iteracao = 0
    concluido = False
    resposta_final = None

    while (
        iteracao < MAX_ITERACOES_POR_TURNO
        and not concluido
    ):
        iteracao += 1

        chamada = encontrar_function_call(
            interaction
        )

        if chamada is None:
            resposta_final = (
                interaction.output_text or ""
            ).strip()

            sessao["previous_interaction_id"] = (
                interaction.id
            )

            concluido = True
            break

        logging.info(
            "Tool solicitada: %s",
            chamada.name
        )

        logging.info(
            "Argumentos: %s",
            chamada.arguments
        )

        resultado_tool = executar_tool(
            chamada.name,
            chamada.arguments,
            mensagem_usuario,
            sessao
        )

        sessao["acoes_executadas"].append({
            "turno": sessao["turnos"],
            "tool": chamada.name,
            "argumentos": chamada.arguments,
            "resultado": resultado_tool
        })

        logging.info(
            "Resultado: %s",
            resultado_tool
        )

        interaction = client.interactions.create(
            model=MODELO,
            input=[
                {
                    "type": "function_result",
                    "name": chamada.name,
                    "call_id": chamada.id,
                    "result": [
                        {
                            "type": "text",
                            "text": json.dumps(
                                resultado_tool,
                                ensure_ascii=False
                            )
                        }
                    ]
                }
            ],
            tools=tools,
            previous_interaction_id=interaction.id
        )

    if not concluido:
        logging.warning(
            "Limite de iterações atingido."
        )

        resposta_final = (
            "Não foi possível concluir este turno "
            "dentro do limite de execução."
        )

    return resposta_final

## 9. Teste 1 — fluxo válido

O cliente fornece todos os dados necessários. O agente deve consultar preço e disponibilidade e, antes de registrar, pedir confirmação.

Depois da confirmação explícita, o Python permite o registro.

In [ ]:
pre_agendamentos.clear()
sessao_1 = nova_sessao()

resposta = atender(
    (
        "Quero banho e hidratação para a Luna. "
        "Ela é uma gata de porte pequeno. "
        "Quero sábado de manhã."
    ),
    sessao_1
)

print("ASSISTENTE:")
print(resposta)

print("\nPré-agendamentos antes da confirmação:")
print(pre_agendamentos)

In [ ]:
resposta = atender(
    "Confirmo. Os dados estão corretos e pode registrar.",
    sessao_1
)

print("ASSISTENTE:")
print(resposta)

print("\nPré-agendamentos depois da confirmação:")
print(pre_agendamentos)

### Ficha estruturada

O registro final também pode ser novamente validado pelo contrato `FichaAtendimento`.

In [ ]:
if pre_agendamentos:
    ficha = FichaAtendimento.model_validate(
        pre_agendamentos[-1]
    )

    print(
        ficha.model_dump_json(
            indent=2
        )
    )

## 10. Teste 2 — horário indisponível

Quarta-feira de manhã está indisponível. O agente não deve registrar o atendimento e deve pedir outra opção ao cliente.

In [ ]:
pre_agendamentos.clear()
sessao_2 = nova_sessao()

resposta = atender(
    (
        "Quero banho para o Thor, um cão de porte médio, "
        "na quarta-feira de manhã."
    ),
    sessao_2
)

print("ASSISTENTE:")
print(resposta)

print("\nPré-agendamentos:")
print(pre_agendamentos)

## 11. Teste 3 — informação faltando

O agente não deve adivinhar o porte do pet.

In [ ]:
pre_agendamentos.clear()
sessao_3 = nova_sessao()

resposta = atender(
    (
        "Quero marcar banho para o Bob, meu cachorro, "
        "no sábado de manhã."
    ),
    sessao_3
)

print("ASSISTENTE:")
print(resposta)

## 12. Teste 4 — relato de problema de saúde

A entrada é interrompida antes do fluxo de pré-agendamento. Nenhuma tool deve ser necessária.

In [ ]:
pre_agendamentos.clear()
sessao_4 = nova_sessao()

resposta = atender(
    (
        "Meu cachorro está apático desde ontem, "
        "mas queria aproveitar e marcar uma tosa para sábado."
    ),
    sessao_4
)

print("ASSISTENTE:")
print(resposta)

print("\nAções executadas:")
print(sessao_4["acoes_executadas"])

## 13. Teste direto do controle de confirmação

Este teste não passa pela LLM. Ele mostra que uma chamada estruturalmente válida ainda pode ser recusada pelo Python quando o cliente não confirmou o registro.

In [ ]:
sessao_controle = nova_sessao()

tentativa = executar_tool(
    "registrar_pre_agendamento",
    {
        "nome_pet": "Luna",
        "especie": "gato",
        "porte": "pequeno",
        "servicos": ["banho"],
        "dia": "sabado",
        "periodo": "manha",
        "valor_total": 45.0
    },
    mensagem_usuario="Quero fazer um agendamento.",
    sessao=sessao_controle
)

print(tentativa)

## 14. O que mudou em relação ao agente de manutenção?

A arquitetura principal foi reaproveitada:

- tools descritas para a LLM;
- contratos Pydantic;
- allowlist;
- executor controlado;
- `function_result`;
- decisões sucessivas;
- estado da sessão;
- condição de parada;
- logs.

O que mudou foi o **domínio** e, com ele, as regras de negócio:

- orçamento deve vir da tabela de preços;
- disponibilidade deve vir da agenda;
- o registro só ocorre após confirmação explícita;
- preço e disponibilidade são revalidados pelo Python;
- relatos de problema de saúde interrompem o fluxo estético.

Esse é o ponto central da atividade: a arquitetura de agente pode ser reaproveitada, mas as capacidades e as regras de controle precisam representar o novo problema.